# Ames Housing Price Prediction Project

## An Elementary, Instructive Jupyter Notebook

**Project question:** What factors help predict the sale price of a house?

This notebook is designed as a beginner-friendly version of the Ames Housing analysis. It explains the full data science workflow step by step:

1. Define the problem
2. Load and inspect the data
3. Explore important variables visually
4. Prepare the data for modeling
5. Build regression models
6. Evaluate model performance
7. Interpret the most important predictors
8. Write a short conclusion

The project uses the Ames Housing dataset, where the target variable is `SalePrice`.

## 1. Project Background

Housing prices are influenced by many features. Some features are physical, such as house size, garage area, number of bathrooms, and basement size. Other features are more qualitative, such as overall quality, condition, age, and neighborhood.

The goal of this project is not only to predict sale price, but also to understand which variables appear most useful for prediction.

From the earlier analysis, the strongest signals were:

- `OverallQual`, which measures overall material and finish quality
- `GrLivArea`, which measures above-ground living area
- Garage-related variables such as `GarageCars` and `GarageArea`
- Basement size, especially `TotalBsmtSF`
- Neighborhood effects

The earlier full analysis found that the Random Forest model performed best, with an R² of about **0.891**, RMSE of about **$28,931**, and MAE of about **$17,465**.

## 2. Import Libraries

We begin by importing the basic Python libraries needed for data analysis, visualization, and machine learning.

- `pandas` handles tables of data
- `numpy` supports numerical calculations
- `matplotlib` creates charts
- `scikit-learn` builds and evaluates machine learning models

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## 3. Load the Data

This notebook tries to load the data in a practical order:

1. If `train.csv` is in the same folder, it loads that file.
2. If `AmesHousing.csv` is in the same folder, it loads that file.
3. If neither local file is available, it tries to load the dataset from OpenML.

For most students, the easiest option is to place the Kaggle `train.csv` file in the same folder as this notebook.

In [ ]:
def load_ames_data():
    if os.path.exists("train.csv"):
        df_local = pd.read_csv("train.csv")
        return df_local, "Local file: train.csv"

    if os.path.exists("AmesHousing.csv"):
        df_local = pd.read_csv("AmesHousing.csv")
        return df_local, "Local file: AmesHousing.csv"

    try:
        from sklearn.datasets import fetch_openml
        data = fetch_openml(name="house_prices", as_frame=True)
        return data.frame.copy(), "OpenML dataset: house_prices"
    except Exception as e:
        raise FileNotFoundError(
            "Could not load the Ames Housing data. Place train.csv or AmesHousing.csv in the same folder as this notebook."
        )


df, data_source = load_ames_data()
print("Data source:", data_source)
print("Rows and columns:", df.shape)
df.head()

## 4. First Look at the Dataset

Before building a model, we need to understand the structure of the data.

Useful questions:

- How many rows and columns are there?
- What are the column names?
- Is the target variable `SalePrice` present?
- Which columns are numeric and which are categorical?

In [ ]:
print("Dataset shape:", df.shape)
print("
First 20 columns:")
print(df.columns[:20].tolist())

print("
Is SalePrice available?", "SalePrice" in df.columns)

print("
Data types:")
print(df.dtypes.value_counts())

## 5. Basic Summary Statistics

The `describe()` function gives a quick numerical summary. It shows statistics such as mean, standard deviation, minimum, maximum, and quartiles.

This helps us understand the scale of variables like sale price, living area, lot area, and garage area.

In [ ]:
df.describe().T.head(20)

## 6. Missing Values

Real-world datasets often contain missing values. In housing data, missing values can happen for many reasons. For example, a missing garage variable may mean that the house has no garage.

For modeling, we will use a pipeline that fills missing numeric values with the median and missing categorical values with the word `None`.

In [ ]:
missing_count = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing_count / len(df) * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
})

missing_table[missing_table["missing_count"] > 0].head(20)

# Exploratory Data Analysis

Exploratory Data Analysis, or EDA, helps us understand the data before modeling. The goal is to see patterns, relationships, and possible problems.

## 7. Distribution of Sale Price

A histogram shows how sale prices are distributed. Housing prices are often right-skewed because a small number of expensive homes stretch the upper end of the distribution.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["SalePrice"].dropna(), bins=35)
plt.title("Distribution of Sale Prices")
plt.xlabel("Sale Price")
plt.ylabel("Number of Houses")
plt.tight_layout()
plt.show()

## 8. Living Area and Sale Price

A scatterplot helps us see whether larger homes tend to sell for higher prices. We expect a positive relationship between `GrLivArea` and `SalePrice`.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["GrLivArea"], df["SalePrice"], alpha=0.6)
plt.title("Above-Ground Living Area vs. Sale Price")
plt.xlabel("Above-Ground Living Area")
plt.ylabel("Sale Price")
plt.tight_layout()
plt.show()

## 9. Overall Quality and Sale Price

`OverallQual` was the most important predictor in the earlier analysis. This makes sense because the quality of materials and finish strongly affects what buyers are willing to pay.

In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(df["OverallQual"], df["SalePrice"], alpha=0.6)
plt.title("Overall Quality vs. Sale Price")
plt.xlabel("Overall Quality")
plt.ylabel("Sale Price")
plt.tight_layout()
plt.show()

## 10. Sale Price by Neighborhood

Neighborhood is a categorical variable. A boxplot is useful because it shows how sale prices differ across groups.

Some neighborhoods tend to have higher median prices than others. This does not mean neighborhood alone determines price, but it often captures location, demand, convenience, and buyer preferences.

In [ ]:
if "Neighborhood" in df.columns:
    order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
    plt.figure(figsize=(12, 6))
    df.boxplot(column="SalePrice", by="Neighborhood", rot=90, grid=False)
    plt.title("Sale Price by Neighborhood")
    plt.suptitle("")
    plt.xlabel("Neighborhood")
    plt.ylabel("Sale Price")
    plt.tight_layout()
    plt.show()

## 11. Correlation With Sale Price

Correlation measures the strength of a linear relationship between two numerical variables.

A correlation close to 1 means a strong positive relationship. A correlation close to 0 means little linear relationship. A correlation close to -1 means a strong negative relationship.

In the earlier analysis, `OverallQual` had the highest correlation with sale price, followed by `GrLivArea`, `GarageCars`, `GarageArea`, and `TotalBsmtSF`.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
correlations = numeric_df.corr()["SalePrice"].drop("SalePrice").sort_values(ascending=False)

top_corr = correlations.head(15)
pd.DataFrame({
    "feature": top_corr.index,
    "correlation_with_sale_price": top_corr.values
})

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(top_corr.sort_values().index, top_corr.sort_values().values)
plt.title("Top Numerical Correlations With Sale Price")
plt.xlabel("Correlation")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Modeling

Now we build models to predict `SalePrice`.

We will compare three models:

1. **Linear Regression:** A simple and interpretable baseline model
2. **Ridge Regression:** A regularized linear model that is more stable when there are many predictors
3. **Random Forest Regression:** A nonlinear model that can capture more complex relationships

## 12. Choose Predictors and Target

The target variable is the value we want to predict. Here, the target is `SalePrice`.

The predictors are the variables used to make the prediction. We remove identifiers such as `Id` because they do not represent meaningful house characteristics.

In [ ]:
target = "SalePrice"
columns_to_drop = ["SalePrice", "Id", "PID"]

X = df.drop(columns=[col for col in columns_to_drop if col in df.columns])
y = df[target]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Number of numeric predictors:", len(numeric_features))
print("Number of categorical predictors:", len(categorical_features))
print("
Sample numeric predictors:", numeric_features[:10])
print("Sample categorical predictors:", categorical_features[:10])

## 13. Train-Test Split

We split the data into two parts:

- Training data: used to fit the model
- Test data: used to evaluate the model on data it has not seen before

This helps us check whether the model generalizes beyond the data used for training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

## 14. Preprocessing Pipeline

Machine learning models need clean numeric input.

For numeric variables, we will:

- Fill missing values with the median
- Standardize the variables for linear models

For categorical variables, we will:

- Fill missing values with `None`
- Convert categories into dummy variables using one-hot encoding

A pipeline keeps these steps organized and prevents data leakage.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

## 15. Build and Compare Models

We now train each model and evaluate it using three metrics:

- **R²:** Measures how much variation in sale price the model explains. Higher is better.
- **RMSE:** Root Mean Squared Error. Lower is better. Larger errors are penalized more strongly.
- **MAE:** Mean Absolute Error. Lower is better. This is easier to interpret as the average absolute prediction error.

In [ ]:
def evaluate_model(model_name, model, X_train, X_test, y_train, y_test):
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    r2 = r2_score(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions, squared=False)
    mae = mean_absolute_error(y_test, predictions)

    return pipe, {
        "model": model_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    }

models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=10.0),
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    )
}

fitted_models = {}
results = []

for model_name, model in models.items():
    fitted_pipe, model_result = evaluate_model(
        model_name, model, X_train, X_test, y_train, y_test
    )
    fitted_models[model_name] = fitted_pipe
    results.append(model_result)

results_df = pd.DataFrame(results).sort_values(by="RMSE")
results_df

## 16. Interpreting the Model Results

The earlier full analysis produced the following model comparison:

| Model | R² | RMSE | MAE |
|---|---:|---:|---:|
| Random Forest | 0.891 | 28,931 | 17,465 |
| Ridge Regression | 0.878 | 30,535 | 19,177 |
| Linear Regression | 0.443 | 65,335 | 21,107 |

The Random Forest model performed best because housing prices often depend on nonlinear relationships and interactions. For example, size matters, but the value of size can depend on quality, neighborhood, and house condition.

## 17. Actual vs. Predicted Sale Prices

This chart compares the Random Forest predictions with the true sale prices. If the model were perfect, all points would fall on the diagonal line.

In [ ]:
best_model = fitted_models["Random Forest"]
rf_predictions = best_model.predict(X_test)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, rf_predictions, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()])
plt.title("Actual vs. Predicted Sale Prices")
plt.xlabel("Actual Sale Price")
plt.ylabel("Predicted Sale Price")
plt.tight_layout()
plt.show()

## 18. Feature Importance

Feature importance helps us understand which variables the Random Forest used most heavily.

In the earlier full analysis, the most important feature was `OverallQual`, followed by `GrLivArea`. This is very reasonable because quality and usable living space are two major drivers of home value.

In [ ]:
rf_model = best_model.named_steps["model"]
fitted_preprocessor = best_model.named_steps["preprocessor"]

feature_names = []
feature_names.extend(numeric_features)

if len(categorical_features) > 0:
    ohe = fitted_preprocessor.named_transformers_["cat"].named_steps["onehot"]
    encoded_names = ohe.get_feature_names_out(categorical_features).tolist()
    feature_names.extend(encoded_names)

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": rf_model.feature_importances_
}).sort_values(by="importance", ascending=False)

importance_df.head(20)

In [ ]:
top_importance = importance_df.head(15).sort_values(by="importance")

plt.figure(figsize=(9, 6))
plt.barh(top_importance["feature"], top_importance["importance"])
plt.title("Top Random Forest Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Final Interpretation

The analysis shows that housing prices are not determined by one factor alone. However, some variables are clearly more informative than others.

The most important predictor was `OverallQual`, which measures the overall material and finish quality of the house. This makes practical sense because buyers are usually willing to pay more for a house that is better built and better maintained.

House size was also very important. The variable `GrLivArea` had a strong positive relationship with sale price, showing that larger living areas generally lead to higher prices.

Neighborhood also mattered because location affects demand, convenience, access to amenities, and buyer preferences. The boxplot showed that sale prices differ noticeably across neighborhoods.

The Random Forest model performed best because it can capture nonlinear patterns and interactions among variables. Ridge Regression also performed well and is easier to interpret than Random Forest. Linear Regression was useful as a baseline, but it performed worse than the other models.

# Short Student Conclusion

This project used the Ames Housing dataset to study what drives home sale prices. The analysis showed that overall quality, living area, garage features, basement size, and neighborhood were among the strongest predictors. The Random Forest model produced the best results, with an R² close to 0.89 in the earlier full analysis. This means the model explained a large portion of the variation in sale prices.

The project also shows why prediction should be combined with interpretation. A model can estimate house prices, but the more useful insight is understanding why certain houses are more valuable. In this dataset, quality, size, and location were the most important themes.

# Reflection Questions

Use these questions to check your understanding:

1. Why is `OverallQual` such a strong predictor of sale price?
2. Why might neighborhood matter even if two houses have the same size?
3. Why do we split the data into training and test sets?
4. Why does Random Forest often perform better than simple Linear Regression?
5. What are the limitations of using historical housing data to predict future prices?

# Optional Extension Ideas

To make the project stronger, you could try the following:

- Use log-transformed sale price as the target
- Remove extreme outliers in `GrLivArea`
- Compare more models, such as Gradient Boosting or XGBoost
- Tune Random Forest hyperparameters
- Create a simpler model using only the top 10 features
- Add clearer business recommendations for buyers, sellers, or real estate agents

# Save Results

The following cell saves the model results and feature importance table as CSV files.

In [ ]:
results_df.to_csv("ames_model_results_elementary.csv", index=False)
importance_df.to_csv("ames_feature_importance_elementary.csv", index=False)

print("Saved files:")
print("ames_model_results_elementary.csv")
print("ames_feature_importance_elementary.csv")